In [1]:
print('Hello world')

Hello world


In [2]:
import gzip
from itertools import product

import urllib.request
MULTI30K_URL = (
    "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/{name}.{lang}.gz"
)
MULTI30K_RAW_NAMES = {"train": "train", "val": "val", "test": "test_2016_flickr"}
lang = ["en", "de"]

for (split,name), lang in product(MULTI30K_RAW_NAMES.items(), ["en", "de"]):
    url = MULTI30K_URL.format(name = name, lang = lang)
    with urllib.request.urlopen(url, timeout = 30) as resp:
        text = gzip.decompress(data = resp.read()).decode(encoding='utf-8')
        print(text.splitlines()[:5])


['Two young, White males are outside near many bushes.', 'Several men in hard hats are operating a giant pulley system.', 'A little girl climbing into a wooden playhouse.', 'A man in a blue shirt is standing on a ladder cleaning a window.', 'Two men are at the stove preparing food.']
['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.', 'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.', 'Ein kleines Mädchen klettert in ein Spielhaus aus Holz.', 'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.', 'Zwei Männer stehen am Herd und bereiten Essen zu.']
['A group of men are loading cotton onto a truck', 'A man sleeping in a green room on a couch.', "A boy wearing headphones sits on a woman's shoulders.", 'Two men setting up a blue ice fishing hut on an iced over lake', 'A balding man wearing a red life jacket is sitting in a small boat.']
['Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen', 'Ein Mann schläft in einem grünen

In [3]:
from pathlib import Path

from training.common import bpe


MULTI30K_URL = (
    "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/{name}.{lang}.gz"
)
MULTI30K_RAW_NAMES = {"train": "train"}
lang = ["en", "de"]
path_start = '/tmp'
train_stats = {}
for (split,name), lang in product(MULTI30K_RAW_NAMES.items(), ["en", "de"]):
    url = MULTI30K_URL.format(name = name, lang = lang)
    with urllib.request.urlopen(url, timeout = 30) as resp:
        text = gzip.decompress(data = resp.read()).decode(encoding='utf-8').splitlines()
        path = Path(f'{path_start}/{name}/{lang}_text.txt')
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text('\n'.join(text))
        for num_merges in [500, 1000, 2000,3000, 5000, 8000, 10000]:
            merges, vocab = bpe.train(text='\n'.join(text), num_merges=num_merges)
            train_stats[f'{name}_{lang}_{str(num_merges)}'] = (merges,vocab)
        
        

In [4]:
import numpy as np

PERCENTILES = [50, 90, 95, 99, 99.9]
# encode rebuilds its merge-rank table on every call, so encoding one row at a time
# is slow: all 29k rows takes ~3 min per config at 5000 merges. Sample for a quick look.
SAMPLE = 5000  # None = every row

rng = np.random.default_rng(seed=0)
rows_by_lang = {}
for lang in ["en", "de"]:
    rows = Path(f"{path_start}/train/{lang}_text.txt").read_text(encoding="utf-8").splitlines()
    if SAMPLE is not None:
        rows = [rows[i] for i in rng.choice(len(rows), size=SAMPLE, replace=False)]
    rows_by_lang[lang] = rows  # same sample for every num_merges, so rows compare


def as_bpe_sees_it(text):
    """bpe maps each whitespace char to one space and keeps runs, so do the same."""
    return "".join(" " if c.isspace() else c for c in text)


header = ["config", "merges", "vocab", "tok/word", "mean"] + [f"p{p}" for p in PERCENTILES] + ["max", "roundtrip"]
print(("{:>16}" + "{:>9}" * (len(header) - 1)).format(*header))
for key, (merges, vocab) in train_stats.items():
    _, lang, num_merges = key.split("_")
    rows = rows_by_lang[lang]
    ids = [bpe.encode(text=row, merges=merges) for row in rows]
    lens = np.array([len(x) for x in ids])
    words = np.array([len(row.split()) for row in rows])
    roundtrip = np.mean(
        [bpe.decode(ids=x, vocab=vocab) == as_bpe_sees_it(text=row) for x, row in zip(ids, rows)]
    )
    stats = [len(merges), len(vocab), lens.sum() / words.sum(), lens.mean()]
    stats += list(np.percentile(lens, PERCENTILES)) + [lens.max(), roundtrip]
    print(f"{key:>16}" + "".join(f"{s:>9.2f}" for s in stats))

          config   merges    vocab tok/word     mean      p50      p90      p95      p99    p99.9      maxroundtrip
    train_en_500   500.00   760.00     1.80    21.43    20.00    31.00    35.00    45.00    60.00    81.00     1.00
   train_en_1000  1000.00  1260.00     1.52    18.16    17.00    26.00    30.00    37.00    48.00    65.00     1.00
   train_en_2000  2000.00  2260.00     1.34    15.96    15.00    23.00    26.00    33.00    42.00    59.00     1.00
   train_en_3000  3000.00  3260.00     1.27    15.10    14.00    21.00    24.00    31.00    38.00    56.00     1.00
   train_en_5000  5000.00  5260.00     1.21    14.38    14.00    20.00    23.00    28.01    37.00    43.00     1.00
   train_en_8000  8000.00  8260.00     1.17    13.91    13.00    19.00    22.00    27.00    35.00    38.00     1.00
  train_en_10000 10000.00 10260.00     1.16    13.76    13.00    19.00    21.00    27.00    34.00    36.00     1.00
    train_de_500   500.00   760.00     2.22    24.62    23.00    37.00  

In [1]:
from training.transformer.mask import causal_keep_mask


print(causal_keep_mask(seq_len=10))

tensor([[[[ True, False, False, False, False, False, False, False, False, False],
          [ True,  True, False, False, False, False, False, False, False, False],
          [ True,  True,  True, False, False, False, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False, False, False],
          [ True,  True,  True,  True,  True, False, False, False, False, False],
          [ True,  True,  True,  True,  True,  True, False, False, False, False],
          [ True,  True,  True,  True,  True,  True,  True, False, False, False],
          [ True,  True,  True,  True,  True,  True,  True,  True, False, False],
          [ True,  True,  True,  True,  True,  True,  True,  True,  True, False],
          [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True]]]])


In [3]:
import torch


ten = torch.zeros((1,2,3))
print(ten.shape[1])

2
